In [ ]:
!pip install -q pandas numpy scikit-learn joblib streamlit plotly

In [ ]:
from google.colab import files
import pandas as pd
import io
import zipfile

uploaded = files.upload()

# Get the uploaded filename
filename = next(iter(uploaded))
file_bytes = io.BytesIO(uploaded[filename])

# Check if the uploaded file is a zip archive
if filename.endswith('.zip'):
    with zipfile.ZipFile(file_bytes) as z:
        # List all files inside the zip archive
        file_list = z.namelist()
        print("Files in zip archive:", file_list)

        # Find the first CSV file inside the zip
        csv_files = [f for f in file_list if f.endswith('.csv')]
        if csv_files:
            # Read the target CSV from the zip
            target_csv = csv_files[0]
            print(f"Reading '{target_csv}' from zip...")
            df = pd.read_csv(z.open(target_csv))
        else:
            raise ValueError("No CSV files found inside the uploaded zip archive.")
else:
    # Read directly if it is a normal CSV file
    df = pd.read_csv(file_bytes)

print("\nDataset loaded successfully!")
print("File name:", filename)
print("Number of rows and columns:", df.shape)

display(df.head())

Saving green-grid-forecasting-daily-household-energy-consumption.zip to green-grid-forecasting-daily-household-energy-consumption (1).zip
Files in zip archive: ['sample_submission.csv', 'test.csv', 'train.csv']
Reading 'sample_submission.csv' from zip...

Dataset loaded successfully!
File name: green-grid-forecasting-daily-household-energy-consumption (1).zip
Number of rows and columns: (21000, 2)


,row_id,kwh
0,59,25.26
1,60,25.26
2,61,25.26
3,62,25.26
4,63,25.26


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDataset information:")
df.info()

Column names:
['row_id', 'kwh']

Data types:
row_id      int64
kwh       float64
dtype: object

Missing values in each column:
row_id    0
kwh       0
dtype: int64

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21000 entries, 0 to 20999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   row_id  21000 non-null  int64  
 1   kwh     21000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 328.3 KB


In [ ]:
# Work on a copy so the original data remains unchanged
data = df.copy()

# Remove spaces from column names
data.columns = data.columns.astype(str).str.strip()

# Remove completely empty columns
data = data.dropna(axis=1, how="all")

print("Available columns:")
for column in data.columns:
    print("-", column)

Available columns:
- row_id
- kwh


In [ ]:
# CHANGE THIS to the exact consumption column name in your dataset
TARGET_COLUMN = "kwh"

assert TARGET_COLUMN in data.columns, (
    f"Target not found. Available columns: {data.columns.tolist()}"
)

print("Target column:", TARGET_COLUMN)

Target column: kwh


In [ ]:
import numpy as np

data = data.dropna(subset=[TARGET_COLUMN]).copy()

# The target must be numeric for regression
data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN], errors="coerce"
)
data = data.dropna(subset=[TARGET_COLUMN])

X = data.drop(columns=[TARGET_COLUMN]).copy()
y = data[TARGET_COLUMN].copy()

# Convert recognizable date/time columns into useful features
for column in list(X.columns):
    name = column.lower()

    if any(word in name for word in ["datetime", "timestamp", "date", "time"]):
        parsed = pd.to_datetime(X[column], errors="coerce")

        if parsed.notna().mean() >= 0.8:
            X[column + "_hour"] = parsed.dt.hour
            X[column + "_day"] = parsed.dt.day
            X[column + "_month"] = parsed.dt.month
            X[column + "_weekday"] = parsed.dt.dayofweek
            X = X.drop(columns=[column])

# Remove rows with invalid numeric targets and align X
X = X.loc[y.index]
y = y.loc[X.index]

# Remove columns with no usable values
X = X.dropna(axis=1, how="all")

# Remove duplicate column names if any exist
X = X.loc[:, ~X.columns.duplicated()]

print("Features used for prediction:")
print(X.columns.tolist())

print("Number of usable rows:", len(X))
print("Target summary:")
display(y.describe())

Features used for prediction:
['row_id']
Number of usable rows: 21000
Target summary:


,kwh
count,2.100000e+04
mean,2.526000e+01
std,6.444776e-12
min,2.526000e+01
25%,2.526000e+01
50%,2.526000e+01
75%,2.526000e+01
max,2.526000e+01


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 16800
Testing rows: 4200


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Numeric features:", numeric_columns)
print("Categorical features:", categorical_columns)

Numeric features: ['row_id']
Categorical features: []


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

results = {}
trained_pipelines = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    results[name] = {
        "MAE": mae,
        "RMSE": rmse
    }

    trained_pipelines[name] = pipeline

    print("\nModel:", name)
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))

results_df = pd.DataFrame(results).T
display(results_df)


Model: Linear Regression
MAE: 0.0
RMSE: 0.0

Model: Gradient Boosting
MAE: 0.0
RMSE: 0.0


,MAE,RMSE
Linear Regression,7.105427e-15,7.105427e-15
Gradient Boosting,7.105427e-15,7.105427e-15


In [ ]:
import joblib
import json
from pathlib import Path

# Choose the model with the lowest test RMSE
best_model_name = min(
    results,
    key=lambda name: results[name]["RMSE"]
)

best_model = trained_pipelines[best_model_name]

print("Selected model:", best_model_name)
print("Test MAE:", results[best_model_name]["MAE"])
print("Test RMSE:", results[best_model_name]["RMSE"])

# Save the complete pipeline, including preprocessing
joblib.dump(best_model, "energy_model.joblib")

# Save the feature schema needed by the website
metadata = {
    "target_column": TARGET_COLUMN,
    "model_name": best_model_name,
    "feature_columns": X.columns.tolist(),
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "metrics": results[best_model_name],
    "numeric_defaults": {},
    "categorical_options": {}
}

for column in numeric_columns:
    metadata["numeric_defaults"][column] = {
        "median": float(X_train[column].median()),
        "minimum": float(X_train[column].min()),
        "maximum": float(X_train[column].max())
    }

for column in categorical_columns:
    values = X_train[column].dropna().astype(str).value_counts()
    metadata["categorical_options"][column] = values.head(20).index.tolist()

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model and metadata saved successfully!")

Selected model: Linear Regression
Test MAE: 7.105427357601002e-15
Test RMSE: 7.105427357601002e-15
Model and metadata saved successfully!


In [ ]:
metadata["target_median"] = float(y_train.median())

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Reference consumption:", metadata["target_median"])

Reference consumption: 25.26


In [ ]:
sample_input = X_test.iloc[[0]]

actual_value = float(y_test.iloc[0])
predicted_value = float(best_model.predict(sample_input)[0])

print("Actual consumption:", round(actual_value, 3))
print("Predicted consumption:", round(predicted_value, 3))
print("Absolute error:", round(abs(actual_value - predicted_value), 3))

Actual consumption: 25.26
Predicted consumption: 25.26
Absolute error: 0.0


In [ ]:

%%writefile app.py
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from pathlib import Path

st.set_page_config(
    page_title="EnergyWise AI",
    page_icon="⚡",
    layout="wide"
)

MODEL_PATH = "energy_model.joblib"
METADATA_PATH = "model_metadata.json"

st.markdown("""
<style>
.main {
    background-color: #f5f8fc;
}
h1 {
    color: #12345a;
}
div[data-testid="stMetric"] {
    background: white;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #e1e8f0;
}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_model():
    model = joblib.load(MODEL_PATH)
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
    return model, metadata

if not Path(MODEL_PATH).exists() or not Path(METADATA_PATH).exists():
    st.error(
        "Model files are missing. Place energy_model.joblib "
        "and model_metadata.json beside app.py."
    )
    st.stop()

model, metadata = load_model()

st.title("⚡ EnergyWise AI")
st.subheader("Energy Consumption Prediction System")

st.write(
    "Enter the available conditions below to estimate energy "
    "consumption using a trained machine-learning model."
)

st.info(
    "This is a data-driven estimate, not a real-time electricity "
    "meter. Accuracy depends on the training dataset."
)

st.sidebar.header("About this project")
st.sidebar.write("Machine Learning: Regression")
st.sidebar.write("Models compared: Linear Regression and Gradient Boosting")
st.sidebar.write(f"Selected model: {metadata['model_name']}")

metrics = metadata.get("metrics", {})

if metrics:
    st.sidebar.metric("Test MAE", f"{metrics['MAE']:.3f}")
    st.sidebar.metric("Test RMSE", f"{metrics['RMSE']:.3f}")

st.header("1. Enter prediction inputs")

feature_columns = metadata["feature_columns"]
numeric_columns = metadata["numeric_columns"]
categorical_columns = metadata["categorical_columns"]

user_values = {}

with st.form("prediction_form"):
    st.write("Complete the fields based on the available information.")

    left, right = st.columns(2)

    for index, column in enumerate(feature_columns):
        target_area = left if index % 2 == 0 else right

        with target_area:
            if column in numeric_columns:
                details = metadata["numeric_defaults"][column]
                default = details["median"]

                minimum = details["minimum"]
                maximum = details["maximum"]

                # Number input allows values beyond training range.
                # Warn the user if the entered value is out of range.
                user_values[column] = st.number_input(
                    column.replace("_", " "),
                    value=float(default),
                    step=0.1,
                    format="%.3f",
                    key=f"numeric_{index}"
                )

            else:
                options = metadata["categorical_options"].get(column, [])
                options = [str(value) for value in options]

                if not options:
                    options = ["Unknown"]

                user_values[column] = st.selectbox(
                    column.replace("_", " "),
                    options=options,
                    key=f"category_{index}"
                )

    submitted = st.form_submit_button(
        "Predict Energy Consumption",
        type="primary",
        use_container_width=True
    )

if submitted:
    input_df = pd.DataFrame(
        [[user_values[column] for column in feature_columns]],
        columns=feature_columns
    )

    # Restore numeric types and categorical types expected by the pipeline
    for column in numeric_columns:
        input_df[column] = pd.to_numeric(input_df[column])

    for column in categorical_columns:
        input_df[column] = input_df[column].astype(str)

    prediction = float(model.predict(input_df)[0])

    if not np.isfinite(prediction):
        st.error("The model returned an invalid prediction.")
        st.stop()

    st.header("2. Prediction result")

    unit_name = (
        "kWh" if "kwh" in metadata["target_column"].lower()
        else "kW" if "kw" in metadata["target_column"].lower()
        else "dataset units"
    )

    first, second = st.columns(2)

    with first:
        st.metric(
            "Predicted consumption",
            f"{prediction:.3f} {unit_name}"
        )

    with second:
        st.metric(
            "Reference consumption (training median)",
            f"{metadata['target_median']:.3f} {unit_name}"
        )

    if prediction > metadata["target_median"]:
        st.warning(
            "The prediction is above the training-data median. "
            "Consider reviewing energy use during this period."
        )
    else:
        st.success(
            "The prediction is at or below the training-data median."
        )

    st.caption(
        "The median is a statistical reference, not a household-specific "
        "efficiency standard. The prediction may be inaccurate for "
        "conditions unlike those in the training data."
    )

    st.header("3. Visual comparison")

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=["Training median", "Predicted consumption"],
        y=[metadata["target_median"], prediction],
        text=[
            f"{metadata['target_median']:.3f}",
            f"{prediction:.3f}"
        ],
        textposition="auto",
        name="Consumption"
    ))

    fig.update_layout(
        title="Predicted consumption vs training-data median",
        xaxis_title="Measurement",
        yaxis_title=f"Consumption ({unit_name})",
        template="plotly_white",
        showlegend=False
    )

    st.plotly_chart(fig, use_container_width=True)

    st.header("4. Input summary")

    summary = pd.DataFrame({
        "Input feature": feature_columns,
        "User-provided value": [
            user_values[column] for column in feature_columns
        ]
    })

    st.dataframe(summary, use_container_width=True, hide_index=True)

    st.download_button(
        "Download prediction report",
        data=summary.to_csv(index=False),
        file_name="energy_prediction_inputs.csv",
        mime="text/csv"
    )

st.divider()

st.caption(
    "EnergyWise AI | Academic AIML project | "
    "Predictions are estimates and should not be used as meter readings."
)


Writing app.py


In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib
plotly

Writing requirements.txt


In [ ]:
from pathlib import Path

for filename in [
    "app.py",
    "requirements.txt",
    "energy_model.joblib",
    "model_metadata.json"
]:
    path = Path(filename)
    print(filename, "—", "READY" if path.exists() else "MISSING")

app.py — READY
requirements.txt — READY
energy_model.joblib — READY
model_metadata.json — READY


In [ ]:
from importlib.metadata import version

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
    "plotly"
]

with open("requirements.txt", "w") as f:
    for package in packages:
        f.write(f"{package}=={version(package)}\n")

print(open("requirements.txt").read())

streamlit==1.66.1
pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
joblib==1.6.0
plotly==5.24.1



In [ ]:
import zipfile

project_files = [
    "app.py",
    "requirements.txt",
    "energy_model.joblib",
    "model_metadata.json"
]

with zipfile.ZipFile(
    "EnergyWise_AI_Project.zip", "w",
    zipfile.ZIP_DEFLATED
) as zipf:
    for filename in project_files:
        zipf.write(filename)

print("Project ZIP created successfully!")

Project ZIP created successfully!


In [ ]:
from google.colab import files

files.download("EnergyWise_AI_Project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>